# FieldVision — Stage 4
## Simulator-Ready Empirical Workload Traces

**Purpose:** convert the measurements from Stages 1–3 into task traces that the new FieldVision simulator can sample directly.

This notebook does **not train RL** and does **not invent measured edge/cloud performance**.

It keeps three categories separate:

1. **Measured data**
   - real input sizes from the datasets;
   - measured preprocessing/inference times from Stage 3;
   - measured output sizes from Stage 3.

2. **Derived systems quantities**
   - idealized transfer time at a configurable reference bandwidth;
   - p50/p95 summaries.

3. **Experimental simulator assumptions**
   - task arrival process;
   - service-class mix;
   - experimental deadline multipliers;
   - number of UAVs.

The experimental assumptions are explicitly labeled so they can later be replaced by collaborator/domain requirements.

## 0. Imports and paths

In [1]:
from pathlib import Path
import json
import math
import random

import numpy as np
import pandas as pd

HOME = Path.home()
PROJECT_ROOT = HOME / "Downloads" / "NOMS_Fieldvision"
OUTPUT_DIR = PROJECT_ROOT / "eda_outputs"
SIM_DIR = PROJECT_ROOT / "simulator_inputs"
SIM_DIR.mkdir(parents=True, exist_ok=True)

STAGE3_RAW = OUTPUT_DIR / "stage3_model_profile_raw.csv"
STAGE3_SUMMARY = OUTPUT_DIR / "stage3_workload_profile_summary.csv"
WORKLOAD_CATALOG = OUTPUT_DIR / "image_workload_catalog.csv"

for p in [STAGE3_RAW, STAGE3_SUMMARY, WORKLOAD_CATALOG]:
    if not p.exists():
        raise FileNotFoundError(
            f"Missing {p}. Run Stages 1–3 first."
        )

print("Project root:", PROJECT_ROOT)
print("Simulator outputs:", SIM_DIR)

Project root: /Users/bwbgv/Downloads/NOMS_Fieldvision
Simulator outputs: /Users/bwbgv/Downloads/NOMS_Fieldvision/simulator_inputs


## 1. Load measured data

In [2]:
stage3 = pd.read_csv(STAGE3_RAW)
stage3_summary = pd.read_csv(STAGE3_SUMMARY)
catalog = pd.read_csv(WORKLOAD_CATALOG)

print("Stage 3 measured profiles:", len(stage3))
print("Stage 3 summary rows:", len(stage3_summary))
print("Image workload catalog rows:", len(catalog))

display(stage3.head())
display(stage3_summary)

Stage 3 measured profiles: 52
Stage 3 summary rows: 4
Image workload catalog rows: 2000


,workload,model,device,path,input_size_mb,width,height,megapixels,decode_ms,model_preprocess_ms,...,extra_peak_cuda_mb,tiles_measured,tiles_total,estimated_full_inference_ms,pretrained_weights,upload_sec_5Mbps,upload_sec_10Mbps,upload_sec_25Mbps,upload_sec_50Mbps,upload_sec_100Mbps
0,Leafy crop classification,MobileNetV3-Small,cpu,/Users/bwbgv/Downloads/NOMS_Fieldvision/datase...,0.006172,39,39,0.001521,18.683667,4.330042,...,NaN,1,1,63.927834,True,0.009875,0.004938,0.001975,0.000988,0.000494
1,Leafy crop classification,MobileNetV3-Small,cpu,/Users/bwbgv/Downloads/NOMS_Fieldvision/datase...,0.006172,39,39,0.001521,6.843667,1.112458,...,NaN,1,1,51.005167,True,0.009875,0.004938,0.001975,0.000988,0.000494
2,Leafy crop classification,MobileNetV3-Small,cpu,/Users/bwbgv/Downloads/NOMS_Fieldvision/datase...,0.006172,39,39,0.001521,2.393250,1.249041,...,NaN,1,1,49.624958,True,0.009875,0.004938,0.001975,0.000988,0.000494
3,Leafy crop classification,MobileNetV3-Small,cpu,/Users/bwbgv/Downloads/NOMS_Fieldvision/datase...,0.006172,39,39,0.001521,1.795750,0.873667,...,NaN,1,1,47.724042,True,0.009875,0.004938,0.001975,0.000988,0.000494
4,Leafy crop classification,MobileNetV3-Small,cpu,/Users/bwbgv/Downloads/NOMS_Fieldvision/datase...,0.006172,39,39,0.001521,1.395292,1.238250,...,NaN,1,1,46.957459,True,0.009875,0.004938,0.001975,0.000988,0.000494


,workload,model,device,samples,median_input_mb,median_megapixels,median_decode_ms,median_preprocess_ms,median_inference_ms,p95_inference_ms,median_output_kb,median_estimated_full_inference_ms
0,Leafy context classification,MobileNetV3-Small,cpu,20,4.000870,1.048576,5.608042,3.580812,48.767854,56.694313,3.90625,48.767854
1,Leafy crop classification,MobileNetV3-Small,cpu,20,0.006172,0.001521,1.654959,0.824896,47.122084,51.651300,3.90625,47.122084
2,Leafy orthomosaic tiled analysis,MobileNetV3-Small tiled proxy,cpu,4,26.450594,16.777216,313.064708,8.511271,246.924354,319.121540,15.62500,2791.581548
3,Rumex weed detection,FasterRCNN-MobileNetV3-320-FPN,cpu,8,5.203753,2.304000,59.755375,4.643250,112.261708,129.435402,0.00000,112.261708


## 2. Map the Stage 2 image catalog into workload classes

We use the larger Stage 2 catalog for **input-size sampling**, because it contains many more real samples than Stage 3.

Stage 3 remains the source for **measured compute/runtime sampling**.

In [3]:
def classify_catalog_row(row):
    dataset = str(row["dataset"])
    path = str(row["source_path"]).lower()

    if dataset == "RumexWeeds":
        return "Rumex weed detection"

    if dataset == "LeafySpurge":
        if "orthomosaic" in path:
            return "Leafy orthomosaic tiled analysis"
        if "context" in path:
            return "Leafy context classification"
        if "crop" in path:
            return "Leafy crop classification"

    return None


catalog["workload"] = catalog.apply(classify_catalog_row, axis=1)
catalog = catalog.dropna(subset=["workload"]).copy()

print("Catalog workload counts:")
display(
    catalog.groupby("workload")
    .size()
    .reset_index(name="samples")
)

Catalog workload counts:


,workload,samples
0,Leafy context classification,346
1,Leafy crop classification,377
2,Leafy orthomosaic tiled analysis,274
3,Rumex weed detection,1000


## 3. Build empirical pools

Each simulated task will draw:

- `input_size_mb` from a real sample in Stage 2;
- `decode_ms`, `preprocess_ms`, `inference_ms`, and `output_kb` from a measured Stage 3 run of the same workload.

This bootstrapping preserves real variability without running the neural networks inside every RL step.

In [4]:
WORKLOADS = sorted(
    set(catalog["workload"].unique())
    .intersection(stage3["workload"].unique())
)

print("Usable workloads:")
for w in WORKLOADS:
    n_input = (catalog["workload"] == w).sum()
    n_compute = (stage3["workload"] == w).sum()
    print(f" - {w}: {n_input} input samples, {n_compute} measured compute samples")

if not WORKLOADS:
    raise RuntimeError("No matching workloads were found between Stage 2 and Stage 3.")

Usable workloads:
 - Leafy context classification: 346 input samples, 20 measured compute samples
 - Leafy crop classification: 377 input samples, 20 measured compute samples
 - Leafy orthomosaic tiled analysis: 274 input samples, 4 measured compute samples
 - Rumex weed detection: 1000 input samples, 8 measured compute samples


## 4. Create measured workload summaries

These are suitable for a methodology/evaluation table because they contain only measured or directly derived statistics.

In [5]:
def p95(series):
    return float(np.percentile(series.dropna(), 95))

summary_rows = []

for workload in WORKLOADS:
    inputs = catalog[catalog["workload"] == workload]
    comp = stage3[stage3["workload"] == workload]

    compute_series = comp["estimated_full_inference_ms"].dropna()

    summary_rows.append({
        "workload": workload,
        "input_samples": len(inputs),
        "compute_samples": len(comp),

        "input_mb_p50": inputs["input_size_mb"].median(),
        "input_mb_p95": p95(inputs["input_size_mb"]),

        "megapixels_p50": inputs["megapixels"].median(),
        "megapixels_p95": p95(inputs["megapixels"]),

        "decode_ms_p50": comp["decode_ms"].median(),
        "preprocess_ms_p50": comp["model_preprocess_ms"].median(),

        "compute_ms_p50": compute_series.median(),
        "compute_ms_p95": p95(compute_series),

        "output_kb_p50": comp["output_kb"].median(),

        "profile_device": ", ".join(sorted(comp["device"].astype(str).unique())),
        "model_proxy": ", ".join(sorted(comp["model"].astype(str).unique())),
    })

measured_summary = pd.DataFrame(summary_rows)

display(measured_summary)

,workload,input_samples,compute_samples,input_mb_p50,input_mb_p95,megapixels_p50,megapixels_p95,decode_ms_p50,preprocess_ms_p50,compute_ms_p50,compute_ms_p95,output_kb_p50,profile_device,model_proxy
0,Leafy context classification,346,20,4.000870,4.000870,1.048576,1.048576,5.608042,3.580812,48.767854,56.694313,3.90625,cpu,MobileNetV3-Small
1,Leafy crop classification,377,20,0.006172,0.006172,0.001521,0.001521,1.654959,0.824896,47.122084,51.651300,3.90625,cpu,MobileNetV3-Small
2,Leafy orthomosaic tiled analysis,274,4,27.505681,33.882896,16.777216,16.777216,313.064708,8.511271,2791.581548,3701.733593,15.62500,cpu,MobileNetV3-Small tiled proxy
3,Rumex weed detection,1000,8,5.205079,5.478160,2.304000,2.304000,59.755375,4.643250,112.261708,129.435402,0.00000,cpu,FasterRCNN-MobileNetV3-320-FPN


## 5. Simulator configuration

**Important:** everything in this cell is an experimental systems assumption, not an agricultural requirement.

We use a balanced workload mix by default because the number of files in a dataset does **not** represent how frequently a real mission generates each workload.

In [6]:
SEED = 42
rng = np.random.default_rng(SEED)

N_TASKS = 5000
N_UAVS = 4

# Development-only task arrival rate across the whole fleet.
TOTAL_ARRIVAL_RATE_TASKS_PER_SEC = 1.0

# Used only to derive an experimental deadline scale.
REFERENCE_BANDWIDTH_MBPS = 25.0

WORKLOAD_PROBS = {
    w: 1.0 / len(WORKLOADS)
    for w in WORKLOADS
}

# Generic systems service classes.
# These are NOT agricultural claims.
SERVICE_CLASSES = {
    "critical": {
        "probability": 0.20,
        "priority_weight": 3,
        "deadline_multiplier": 1.25,
    },
    "standard": {
        "probability": 0.60,
        "priority_weight": 2,
        "deadline_multiplier": 2.0,
    },
    "delay_tolerant": {
        "probability": 0.20,
        "priority_weight": 1,
        "deadline_multiplier": 5.0,
    },
}

print("These are DEVELOPMENT assumptions only:")
print("N_TASKS =", N_TASKS)
print("N_UAVS =", N_UAVS)
print("Arrival rate =", TOTAL_ARRIVAL_RATE_TASKS_PER_SEC, "tasks/s")
print("Reference bandwidth =", REFERENCE_BANDWIDTH_MBPS, "Mbps")

These are DEVELOPMENT assumptions only:
N_TASKS = 5000
N_UAVS = 4
Arrival rate = 1.0 tasks/s
Reference bandwidth = 25.0 Mbps


## 6. Helper functions for empirical task sampling

In [7]:
def sample_service_class(rng):
    names = list(SERVICE_CLASSES.keys())
    probs = np.array(
        [SERVICE_CLASSES[n]["probability"] for n in names],
        dtype=float
    )
    probs = probs / probs.sum()
    return rng.choice(names, p=probs)


def sample_empirical_task(workload, rng):
    input_pool = catalog[catalog["workload"] == workload]
    compute_pool = stage3[stage3["workload"] == workload]

    input_row = input_pool.iloc[
        rng.integers(0, len(input_pool))
    ]

    compute_row = compute_pool.iloc[
        rng.integers(0, len(compute_pool))
    ]

    input_mb = float(input_row["input_size_mb"])
    megapixels = float(input_row["megapixels"])

    compute_ms = float(compute_row["estimated_full_inference_ms"])
    decode_ms = float(compute_row["decode_ms"])
    preprocess_ms = float(compute_row["model_preprocess_ms"])
    output_kb = float(compute_row["output_kb"])

    reference_upload_s = (input_mb * 8.0) / REFERENCE_BANDWIDTH_MBPS

    reference_completion_s = (
        reference_upload_s
        + decode_ms / 1000.0
        + preprocess_ms / 1000.0
        + compute_ms / 1000.0
    )

    service_class = sample_service_class(rng)
    svc = SERVICE_CLASSES[service_class]

    experimental_deadline_s = (
        reference_completion_s
        * svc["deadline_multiplier"]
    )

    return {
        "workload": workload,

        # Measured/empirical
        "input_size_mb": input_mb,
        "megapixels": megapixels,
        "measured_decode_ms": decode_ms,
        "measured_preprocess_ms": preprocess_ms,
        "measured_compute_ms_current_host": compute_ms,
        "measured_output_kb": output_kb,

        # Provenance
        "compute_profile_device": str(compute_row["device"]),
        "compute_profile_model": str(compute_row["model"]),
        "source_dataset": str(input_row["dataset"]),

        # Derived
        "reference_upload_s_at_25mbps": reference_upload_s,
        "reference_completion_s": reference_completion_s,

        # Experimental assumptions
        "service_class": service_class,
        "priority_weight": int(svc["priority_weight"]),
        "experimental_deadline_s": experimental_deadline_s,
    }

## 7. Generate a reusable multi-UAV workload trace

Arrival times use an exponential inter-arrival distribution, giving a simple Poisson arrival process.

The trace is reproducible because the random seed is fixed.

In [8]:
workload_names = list(WORKLOAD_PROBS.keys())
workload_probs = np.array(
    [WORKLOAD_PROBS[w] for w in workload_names],
    dtype=float
)
workload_probs = workload_probs / workload_probs.sum()

interarrival_s = rng.exponential(
    scale=1.0 / TOTAL_ARRIVAL_RATE_TASKS_PER_SEC,
    size=N_TASKS
)

arrival_times_s = np.cumsum(interarrival_s)

tasks = []

for task_idx in range(N_TASKS):
    workload = rng.choice(
        workload_names,
        p=workload_probs
    )

    task = sample_empirical_task(
        workload,
        rng
    )

    task.update({
        "task_id": f"task_{task_idx:06d}",
        "arrival_time_s": float(arrival_times_s[task_idx]),
        "uav_id": int(rng.integers(0, N_UAVS)),
    })

    tasks.append(task)

task_trace = pd.DataFrame(tasks)

first_cols = [
    "task_id",
    "arrival_time_s",
    "uav_id",
    "workload",
    "service_class",
    "priority_weight",
    "experimental_deadline_s",
]

remaining_cols = [
    c for c in task_trace.columns
    if c not in first_cols
]

task_trace = task_trace[first_cols + remaining_cols]

display(task_trace.head(10))

,task_id,arrival_time_s,uav_id,workload,service_class,priority_weight,experimental_deadline_s,input_size_mb,megapixels,measured_decode_ms,measured_preprocess_ms,measured_compute_ms_current_host,measured_output_kb,compute_profile_device,compute_profile_model,source_dataset,reference_upload_s_at_25mbps,reference_completion_s
0,task_000000,2.404209,1,Leafy context classification,delay_tolerant,1,6.697156,4.000870,1.048576,5.536792,3.498250,50.117792,3.90625,cpu,MobileNetV3-Small,LeafySpurge,1.280278,1.339431
1,task_000001,4.740398,2,Leafy orthomosaic tiled analysis,critical,3,17.477765,34.676382,16.777216,75.166375,8.579459,2802.024183,15.62500,cpu,MobileNetV3-Small tiled proxy,LeafySpurge,11.096442,13.982212
2,task_000002,7.125159,2,Leafy orthomosaic tiled analysis,delay_tolerant,1,54.395912,25.624428,16.777216,328.896334,7.505665,2342.963567,15.62500,cpu,MobileNetV3-Small tiled proxy,LeafySpurge,8.199817,10.879182
3,task_000003,7.404954,3,Leafy crop classification,standard,2,0.102975,0.006172,0.001521,1.633709,1.045125,46.833334,3.90625,cpu,MobileNetV3-Small,LeafySpurge,0.001975,0.051487
4,task_000004,7.491391,1,Leafy crop classification,critical,3,0.065461,0.006172,0.001521,1.795750,0.873667,47.724042,3.90625,cpu,MobileNetV3-Small,LeafySpurge,0.001975,0.052369
5,task_000005,8.944051,0,Leafy context classification,critical,3,1.672624,4.000870,1.048576,5.765916,3.705667,48.349417,3.90625,cpu,MobileNetV3-Small,LeafySpurge,1.280278,1.338099
6,task_000006,10.354012,3,Rumex weed detection,critical,3,2.348013,5.184388,2.304000,72.744125,16.938084,129.724167,0.00000,cpu,FasterRCNN-MobileNetV3-320-FPN,RumexWeeds,1.659004,1.878411
7,task_000007,13.478308,3,Leafy context classification,delay_tolerant,1,6.725308,4.000870,1.048576,4.709542,3.374250,56.699458,3.90625,cpu,MobileNetV3-Small,LeafySpurge,1.280278,1.345062
8,task_000008,13.557602,2,Leafy crop classification,standard,2,0.105941,0.006172,0.001521,2.266791,0.741250,47.987542,3.90625,cpu,MobileNetV3-Small,LeafySpurge,0.001975,0.052971
9,task_000009,14.604163,0,Leafy crop classification,critical,3,0.076170,0.006172,0.001521,6.843667,1.112458,51.005167,3.90625,cpu,MobileNetV3-Small,LeafySpurge,0.001975,0.060936


## 8. Sanity checks

In [9]:
print("Tasks per UAV:")
display(
    task_trace.groupby("uav_id")
    .size()
    .reset_index(name="tasks")
)

print("\nTasks per workload:")
display(
    task_trace.groupby("workload")
    .size()
    .reset_index(name="tasks")
)

print("\nTasks per service class:")
display(
    task_trace.groupby("service_class")
    .size()
    .reset_index(name="tasks")
)

print("\nDeadline distributions:")
display(
    task_trace.groupby(
        ["workload", "service_class"]
    )["experimental_deadline_s"]
    .agg(["count", "median", "min", "max"])
)

Tasks per UAV:


,uav_id,tasks
0,0,1228
1,1,1256
2,2,1221
3,3,1295



Tasks per workload:


,workload,tasks
0,Leafy context classification,1258
1,Leafy crop classification,1230
2,Leafy orthomosaic tiled analysis,1287
3,Rumex weed detection,1225



Tasks per service class:


,service_class,tasks
0,critical,980
1,delay_tolerant,1025
2,standard,2995



Deadline distributions:


count     median        min  \
workload                         service_class                                 
Leafy context classification     critical          271   1.673394   1.669769   
                                 delay_tolerant    239   6.693574   6.679074   
                                 standard          748   2.677430   2.671630   
Leafy crop classification        critical          232   0.064458   0.062639   
                                 delay_tolerant    277   0.257830   0.250557   
                                 standard          721   0.103066   0.100223   
Leafy orthomosaic tiled analysis critical          249  14.992377   4.413337   
                                 delay_tolerant    249  60.175879  16.126897   
                                 standard          789  24.007810   6.246440   
Rumex weed detection             critical          228   2.306233   2.141019   
                                 delay_tolerant    260   9.246270   8.627350   
                                 standard          737   3.694899   3.407145   

                                                       max  
workload                         service_class              
Leafy context classification     critical         1.684413  
                                 delay_tolerant   6.737654  
                                 standard         2.695061  
Leafy crop classification        critical         0.111146  
                                 delay_tolerant   0.444583  
                                 standard         0.177833  
Leafy orthomosaic tiled analysis critical        19.081652  
                                 delay_tolerant  77.124621  
                                 standard        30.730560  
Rumex weed detection             critical         2.463198  
                                 delay_tolerant   9.828276  
                                 standard         3.987441

## 9. Add normalized compute/network demand

These are derived features for the simulator, not new measurements.

In [10]:
global_compute_median = task_trace[
    "measured_compute_ms_current_host"
].median()

task_trace["compute_demand_normalized"] = (
    task_trace["measured_compute_ms_current_host"]
    / global_compute_median
)

task_trace["network_demand_normalized"] = (
    task_trace["input_size_mb"]
    / task_trace["input_size_mb"].median()
)

display(
    task_trace[[
        "task_id",
        "workload",
        "input_size_mb",
        "measured_compute_ms_current_host",
        "network_demand_normalized",
        "compute_demand_normalized",
    ]].head(10)
)

,task_id,workload,input_size_mb,measured_compute_ms_current_host,network_demand_normalized,compute_demand_normalized
0,task_000000,Leafy context classification,4.000870,50.117792,1.000000,0.480804
1,task_000001,Leafy orthomosaic tiled analysis,34.676382,2802.024183,8.667211,26.881185
2,task_000002,Leafy orthomosaic tiled analysis,25.624428,2342.963567,6.404714,22.477193
3,task_000003,Leafy crop classification,0.006172,46.833334,0.001543,0.449295
4,task_000004,Leafy crop classification,0.006172,47.724042,0.001543,0.457840
5,task_000005,Leafy context classification,4.000870,48.349417,1.000000,0.463840
6,task_000006,Rumex weed detection,5.184388,129.724167,1.295815,1.244507
7,task_000007,Leafy context classification,4.000870,56.699458,1.000000,0.543946
8,task_000008,Leafy crop classification,0.006172,47.987542,0.001543,0.460368
9,task_000009,Leafy crop classification,0.006172,51.005167,0.001543,0.489317


## 10. Initial compute-tier profile

At this point we have only one measured compute tier: the machine on which Stage 3 was run.

**Do not fabricate UAV, edge, or cloud latencies here.**

Later, rerun Stage 3 on those devices and append the measured profiles.

In [11]:
device_rows = []

for workload in WORKLOADS:
    comp = stage3[stage3["workload"] == workload]

    device_rows.append({
        "workload": workload,
        "compute_tier": "stage3_current_host",
        "device": ", ".join(
            sorted(comp["device"].astype(str).unique())
        ),
        "compute_ms_p50": comp["estimated_full_inference_ms"].median(),
        "compute_ms_p95": np.percentile(
            comp["estimated_full_inference_ms"].dropna(),
            95
        ),
        "is_measured": True,
    })

hardware_profiles = pd.DataFrame(device_rows)

display(hardware_profiles)

,workload,compute_tier,device,compute_ms_p50,compute_ms_p95,is_measured
0,Leafy context classification,stage3_current_host,cpu,48.767854,56.694313,True
1,Leafy crop classification,stage3_current_host,cpu,47.122084,51.651300,True
2,Leafy orthomosaic tiled analysis,stage3_current_host,cpu,2791.581548,3701.733593,True
3,Rumex weed detection,stage3_current_host,cpu,112.261708,129.435402,True


## 11. Save simulator-ready files

Outputs:

- `fieldvision_task_trace.csv`
- `fieldvision_measured_workload_summary.csv`
- `fieldvision_compute_tier_profiles.csv`
- `fieldvision_stage4_config.json`

In [12]:
trace_path = SIM_DIR / "fieldvision_task_trace.csv"
summary_path = SIM_DIR / "fieldvision_measured_workload_summary.csv"
hardware_path = SIM_DIR / "fieldvision_compute_tier_profiles.csv"
config_path = SIM_DIR / "fieldvision_stage4_config.json"

task_trace.to_csv(trace_path, index=False)
measured_summary.to_csv(summary_path, index=False)
hardware_profiles.to_csv(hardware_path, index=False)

config = {
    "seed": SEED,
    "n_tasks": N_TASKS,
    "n_uavs": N_UAVS,
    "total_arrival_rate_tasks_per_sec": TOTAL_ARRIVAL_RATE_TASKS_PER_SEC,
    "reference_bandwidth_mbps": REFERENCE_BANDWIDTH_MBPS,
    "workload_probabilities": WORKLOAD_PROBS,
    "service_classes": SERVICE_CLASSES,
    "important_note": (
        "Arrival rates, service-class probabilities, priority weights, and "
        "deadline multipliers are experimental simulator assumptions. "
        "Input sizes and Stage 3 runtime measurements are empirical."
    ),
}

with open(config_path, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

print("Saved:")
print(" -", trace_path)
print(" -", summary_path)
print(" -", hardware_path)
print(" -", config_path)

Saved:
 - /Users/bwbgv/Downloads/NOMS_Fieldvision/simulator_inputs/fieldvision_task_trace.csv
 - /Users/bwbgv/Downloads/NOMS_Fieldvision/simulator_inputs/fieldvision_measured_workload_summary.csv
 - /Users/bwbgv/Downloads/NOMS_Fieldvision/simulator_inputs/fieldvision_compute_tier_profiles.csv
 - /Users/bwbgv/Downloads/NOMS_Fieldvision/simulator_inputs/fieldvision_stage4_config.json


## 12. Write a tiny Python loader for the future simulator

This creates:

`~/Downloads/NOMS_Fieldvision/simulator_inputs/fieldvision_workload_loader.py`

In [13]:
loader_code = '''
from pathlib import Path
import pandas as pd
import numpy as np


class FieldVisionWorkloadTrace:
    """
    Lightweight replay/sampling interface for Stage 4 task traces.
    """

    def __init__(self, csv_path, seed=42):
        self.csv_path = Path(csv_path)
        self.df = pd.read_csv(self.csv_path)
        self.rng = np.random.default_rng(seed)
        self.reset()

    def reset(self):
        self.index = 0

    def __len__(self):
        return len(self.df)

    def next_task(self):
        # Replay tasks in trace arrival order.
        if self.index >= len(self.df):
            return None

        row = self.df.iloc[self.index]
        self.index += 1
        return row.to_dict()

    def sample_task(self, workload=None, service_class=None):
        # Bootstrap a random task from the empirical trace.
        df = self.df

        if workload is not None:
            df = df[df["workload"] == workload]

        if service_class is not None:
            df = df[df["service_class"] == service_class]

        if len(df) == 0:
            raise ValueError("No matching tasks.")

        idx = self.rng.integers(0, len(df))
        return df.iloc[idx].to_dict()

    def tasks_arriving_between(self, start_s, end_s):
        # Return tasks whose trace arrival falls inside [start_s, end_s).
        subset = self.df[
            (self.df["arrival_time_s"] >= start_s)
            & (self.df["arrival_time_s"] < end_s)
        ]
        return subset.to_dict("records")
'''

loader_path = SIM_DIR / "fieldvision_workload_loader.py"

with open(loader_path, "w", encoding="utf-8") as f:
    f.write(loader_code)

print("Created:", loader_path)

Created: /Users/bwbgv/Downloads/NOMS_Fieldvision/simulator_inputs/fieldvision_workload_loader.py


## 13. Test the simulator loader

In [14]:
import sys

if str(SIM_DIR) not in sys.path:
    sys.path.append(str(SIM_DIR))

from fieldvision_workload_loader import FieldVisionWorkloadTrace

trace = FieldVisionWorkloadTrace(
    SIM_DIR / "fieldvision_task_trace.csv"
)

print("Trace length:", len(trace))

print("\nFirst replayed task:")
print(trace.next_task())

print("\nRandom Rumex task:")
print(
    trace.sample_task(
        workload="Rumex weed detection"
    )
)

print("\nTasks arriving in the first 5 seconds:")
first_window = trace.tasks_arriving_between(0, 5)
print("Count:", len(first_window))
for task in first_window[:3]:
    print(task)

Trace length: 5000

First replayed task:
{'task_id': 'task_000000', 'arrival_time_s': 2.4042086039659947, 'uav_id': 1, 'workload': 'Leafy context classification', 'service_class': 'delay_tolerant', 'priority_weight': 1, 'experimental_deadline_s': 6.697155771567487, 'input_size_mb': 4.0008697509765625, 'megapixels': 1.048576, 'measured_decode_ms': 5.536791999475099, 'measured_preprocess_ms': 3.4982500001206063, 'measured_compute_ms_current_host': 50.11779200140154, 'measured_output_kb': 3.90625, 'compute_profile_device': 'cpu', 'compute_profile_model': 'MobileNetV3-Small', 'source_dataset': 'LeafySpurge', 'reference_upload_s_at_25mbps': 1.2802783203125, 'reference_completion_s': 1.3394311543134974, 'compute_demand_normalized': 0.4808044331530945, 'network_demand_normalized': 1.0}

Random Rumex task:
{'task_id': 'task_000492', 'arrival_time_s': 491.2361502958842, 'uav_id': 0, 'workload': 'Rumex weed detection', 'service_class': 'standard', 'priority_weight': 2, 'experimental_deadline_s':

# What Stage 4 gives us

At this point, the simulator no longer needs to generate generic random chunks.

It can receive tasks containing:

- task ID;
- UAV ID;
- arrival time;
- workload type;
- real sampled input size;
- measured current-host compute demand;
- measured preprocessing demand;
- measured output size;
- service class;
- experimental deadline;
- priority weight.

The next major stage is:

## Stage 5 — UAV-agent, queue-based, shared network/compute simulator

Stage 5 will replace the old resource-as-agent `PROCESS/IGNORE` formulation with:

```
UAV agent
   ↓
choose queued task
   ↓
local / edge / cloud / defer
   ↓
shared wireless transport
   ↓
shared compute queue
   ↓
SLA outcome
```

UAV/edge/cloud hardware runtimes can be added later by rerunning Stage 3 on those devices and extending the compute-tier profile file.